# CBAM Regional YAML Generator

Builds calibrated RICE-N region yamls for three CBAM research setups:

- **`setup_3`** — Minimal: EU | Merged CBAM Exporters | Rest of World  
- **`setup_5`** — Sector-split: EU | Eurasia | China | MENA | RoW  
- **`setup_7`** — Granular: EU | Eurasia | China | India | Gulf | N.Africa | RoW  

**Approach**: merges already-calibrated 20-region yamls from `other_yamls/20_regions/`
using GDP-weighted aggregation. Does **not** re-query the World Bank API.

**Outputs**:
- `cbam_yamls/setup_N/K.yml` — one yaml per CBAM region
- `cbam_yamls/CountryClass_cbam_N.csv` — country→CBAM-region mapping for MRIO aggregation

See `cbam_yamls/README.md` for full documentation of region choices and sector rationale.

## 1. Imports & paths

In [1]:
import os
import sys
import yaml
import json
import numpy as np
import pandas as pd

# resolve repo root regardless of cwd
NOTEBOOK_DIR = os.path.abspath("")
REPO_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", ".."))
sys.path.insert(0, REPO_ROOT)

YAMLS_20_DIR   = os.path.join(REPO_ROOT, "other_yamls", "20_regions")
CBAM_DIR       = os.path.join(REPO_ROOT, "cbam_yamls")
CSV_ASSET_DIR  = os.path.join(REPO_ROOT, "csv_asset")
CC20_PATH      = os.path.join(CSV_ASSET_DIR, "CountryClass_20.csv")
IMPORT_JSON_20 = os.path.join(CSV_ASSET_DIR, "20_import_2016.json")

print("Repo root     :", REPO_ROOT)
print("20-region yamls:", YAMLS_20_DIR)
print("CBAM output dir:", CBAM_DIR)

Repo root     : /Users/pwozny/Repos/phd/climate-cooperation-competition
20-region yamls: /Users/pwozny/Repos/phd/climate-cooperation-competition/other_yamls/20_regions
CBAM output dir: /Users/pwozny/Repos/phd/climate-cooperation-competition/cbam_yamls


## 2. Load calibrated 20-region parameters

In [2]:
def load_20_yamls(yamls_dir):
    """Load all 20 region yamls -> dict {rig_int: param_dict}."""
    params = {}
    for i in range(1, 21):
        path = os.path.join(yamls_dir, f"{i}.yml")
        if not os.path.exists(path):
            print(f"  WARNING: missing {path}")
            continue
        with open(path) as f:
            data = yaml.safe_load(f)
        params[i] = data["_RICE_CONSTANT"]
    print(f"Loaded {len(params)} region yamls.")
    return params

params20 = load_20_yamls(YAMLS_20_DIR)

# Approximate base-year GDP proxy: A * K^gamma * (L/1000)^(1-gamma)
gamma = 0.3
def approx_gdp(p):
    return p["xA_0"] * (p["xK_0"] ** gamma) * (p["xL_0"] / 1000) ** (1 - gamma)

gdp20 = {i: approx_gdp(params20[i]) for i in params20}

print("\nBase-year GDP proxy (trillion USD) by RIG:")
for i in sorted(gdp20):
    print(f"  RIG {i:2d}: GDP={gdp20[i]:7.3f}T  L0={params20[i]['xL_0']:7.1f}M  "
          f"sigma={params20[i]['xsigma_0']:.3f}")

Loaded 20 region yamls.

Base-year GDP proxy (trillion USD) by RIG:
  RIG  1: GDP= 23.315T  L0=  332.0M  sigma=0.216
  RIG  2: GDP=  2.015T  L0=   38.3M  sigma=0.329
  RIG  3: GDP= 18.880T  L0=  502.5M  sigma=0.153
  RIG  4: GDP=  2.956T  L0=  316.8M  sigma=0.903
  RIG  5: GDP=  0.233T  L0=   94.5M  sigma=1.352
  RIG  6: GDP=  8.980T  L0=  223.3M  sigma=0.273
  RIG  7: GDP=  0.844T  L0=  105.8M  sigma=0.639
  RIG  8: GDP= 17.820T  L0= 1412.4M  sigma=0.784
  RIG  9: GDP=  1.935T  L0=  573.6M  sigma=0.627
  RIG 10: GDP=  3.150T  L0= 1407.6M  sigma=0.867
  RIG 11: GDP=  0.818T  L0=  486.1M  sigma=0.409
  RIG 12: GDP=  2.241T  L0=   68.1M  sigma=0.589
  RIG 13: GDP=  0.303T  L0=   65.8M  sigma=0.870
  RIG 14: GDP=  0.952T  L0=  293.0M  sigma=1.210
  RIG 15: GDP=  0.450T  L0=   28.4M  sigma=0.310
  RIG 16: GDP=  3.999T  L0=  546.3M  sigma=0.319
  RIG 17: GDP=  0.115T  L0=   46.8M  sigma=0.386
  RIG 18: GDP=  0.402T  L0=   69.1M  sigma=1.093
  RIG 19: GDP=  0.930T  L0=  519.8M  sigma=0.290
 

## 3. Region merge function

Merges a list of 20-region RIGs into a single CBAM region.

- **Extensive params** (K, L): summed directly
- **TFP** (A): back-calculated from merged Y/K/L so Cobb-Douglas holds exactly
- **Intensive params** (g_A, delta_A, sigma, saving, mitigation): GDP-weighted average
- **ximport**: re-keyed to CBAM region indices; intra-group trade removed; GDP-weighted across sub-regions

In [3]:
def merge_regions(rig_list, params20, gdp20, import_data_20, cbam_index_map):
    """
    Merge a list of 20-region RIGs into a single CBAM region parameter dict.

    Parameters
    ----------
    rig_list        : list[int]  RIG indices to merge (from the 20-region set)
    params20        : dict       {rig: param_dict} for all 20 regions
    gdp20           : dict       {rig: float} approximate base-year GDP
    import_data_20  : dict       nested 20-region import JSON (str keys)
    cbam_index_map  : dict       {rig20_int: cbam_region_int} for this setup
    """
    rigs = [r for r in rig_list if r in params20]
    if not rigs:
        raise ValueError(f"None of {rig_list} found in params20")

    total_gdp = sum(gdp20[r] for r in rigs)
    w = {r: gdp20[r] / total_gdp for r in rigs}   # GDP weights

    # -- Extensive: sum --
    xL_0 = sum(params20[r]["xL_0"] for r in rigs)
    xL_a = sum(params20[r]["xL_a"] for r in rigs)
    xK_0 = sum(params20[r]["xK_0"] for r in rigs)

    # -- Population growth: slope-weighted --
    pop_slope_num = sum(
        (params20[r]["xL_a"] - params20[r]["xL_0"]) * params20[r]["xl_g"] for r in rigs
    )
    pop_slope_den = xL_a - xL_0
    xl_g = pop_slope_num / pop_slope_den if abs(pop_slope_den) > 1e-6 else np.mean([params20[r]["xl_g"] for r in rigs])

    # -- TFP: back-calculate A so Cobb-Douglas holds exactly --
    xA_0 = total_gdp / ((xK_0 ** gamma) * (xL_0 / 1000) ** (1 - gamma))

    # -- GDP-weighted intensive params --
    xg_A          = sum(w[r] * params20[r]["xg_A"]          for r in rigs)
    xdelta_A      = sum(w[r] * params20[r]["xdelta_A"]       for r in rigs)
    xsigma_0      = sum(w[r] * params20[r]["xsigma_0"]       for r in rigs)
    xmitigation_0 = sum(w[r] * params20[r]["xmitigation_0"]  for r in rigs)
    xsaving_0     = sum(params20[r]["xsaving_0"] * gdp20[r]  for r in rigs) / total_gdp
    xexport       = sum(params20[r].get("xexport", 0.0) * gdp20[r] for r in rigs) / total_gdp

    # -- ximport: re-key to CBAM indices, remove intra-group trade --
    merged_cbam_idx = cbam_index_map[rigs[0]]
    all_cbam = sorted(set(cbam_index_map.values()))
    ximport_cbam = {str(c): 0.0 for c in all_cbam}
    ximport_cbam[str(merged_cbam_idx)] = 0.0   # no self-imports

    for r in rigs:
        r_str = str(r)
        if r_str not in import_data_20:
            continue
        for src_str, val in import_data_20[r_str].items():
            src_cbam = cbam_index_map.get(int(src_str))
            if src_cbam is None or src_cbam == merged_cbam_idx:
                continue
            ximport_cbam[str(src_cbam)] += val * w[r]

    return {
        "xA_0":          float(xA_0),
        "xK_0":          float(xK_0),
        "xL_0":          float(xL_0),
        "xL_a":          float(xL_a),
        "xa_1":          0,
        "xa_2":          0.00236,
        "xa_3":          2,
        "xdelta_A":      float(xdelta_A),
        "xg_A":          float(xg_A),
        "xgamma":        0.3,
        "xl_g":          float(xl_g),
        "xmitigation_0": float(xmitigation_0),
        "xsaving_0":     float(xsaving_0),
        "xsigma_0":      float(xsigma_0),
        "xtax":          0.0,
        "xexport":       float(xexport),
        "ximport":       ximport_cbam,
    }

print("merge_regions() defined.")

merge_regions() defined.


## 4. CBAM setup definitions

Three setups of increasing granularity. Each is a list of `(cbam_index, label, [rig20s])`.
Region 1 is always EU; the last region is always Rest of World.

In [4]:
SETUPS = {
    "setup_3": {
        "description": "Minimal: EU | Merged CBAM Exporters | Rest of World",
        "regions": [
            (1, "EU & Western Europe",       [3]),
            (2, "CBAM Exporters (merged)",   [4, 5, 8, 10, 12, 13, 14]),
            (3, "Rest of World",             [1, 2, 6, 7, 9, 11, 15, 16, 17, 18, 19, 20]),
        ],
    },
    "setup_5": {
        "description": "Sector-split: EU | Eurasia | China | MENA | RoW",
        "regions": [
            (1, "EU & Western Europe",       [3]),
            (2, "Russia + Turkey + Eurasia", [4, 5]),
            (3, "China",                     [8]),
            (4, "MENA (Gulf + N.Africa)",    [12, 13, 14]),
            (5, "Rest of World",             [1, 2, 6, 7, 9, 10, 11, 15, 16, 17, 18, 19, 20]),
        ],
    },
    "setup_7": {
        "description": "Granular: EU | Eurasia | China | India | Gulf | N.Africa | RoW",
        "regions": [
            (1, "EU & Western Europe",         [3]),
            (2, "Russia + Turkey + Eurasia",   [4, 5]),
            (3, "China",                       [8]),
            (4, "India",                       [10]),
            (5, "Gulf States",                 [12]),
            (6, "N.Africa + MENA developing",  [13, 14]),
            (7, "Rest of World",               [1, 2, 6, 7, 9, 11, 15, 16, 17, 18, 19, 20]),
        ],
    },
}

def build_cbam_index_map(setup_regions):
    """Map each RIG20 integer to its CBAM region index for a given setup."""
    m = {}
    for cbam_idx, _label, rigs in setup_regions:
        for r in rigs:
            m[r] = cbam_idx
    return m

print("Setup summary:")
for sname, sdef in SETUPS.items():
    print(f"\n  {sname}: {sdef['description']}")
    for cbam_idx, label, rigs in sdef["regions"]:
        print(f"    Region {cbam_idx}. {label:<35} <- RIGs {rigs}")

Setup summary:

  setup_3: Minimal: EU | Merged CBAM Exporters | Rest of World
    Region 1. EU & Western Europe                 <- RIGs [3]
    Region 2. CBAM Exporters (merged)             <- RIGs [4, 5, 8, 10, 12, 13, 14]
    Region 3. Rest of World                       <- RIGs [1, 2, 6, 7, 9, 11, 15, 16, 17, 18, 19, 20]

  setup_5: Sector-split: EU | Eurasia | China | MENA | RoW
    Region 1. EU & Western Europe                 <- RIGs [3]
    Region 2. Russia + Turkey + Eurasia           <- RIGs [4, 5]
    Region 3. China                               <- RIGs [8]
    Region 4. MENA (Gulf + N.Africa)              <- RIGs [12, 13, 14]
    Region 5. Rest of World                       <- RIGs [1, 2, 6, 7, 9, 10, 11, 15, 16, 17, 18, 19, 20]

  setup_7: Granular: EU | Eurasia | China | India | Gulf | N.Africa | RoW
    Region 1. EU & Western Europe                 <- RIGs [3]
    Region 2. Russia + Turkey + Eurasia           <- RIGs [4, 5]
    Region 3. China                          

## 5. Load 20-region trade data

In [5]:
with open(IMPORT_JSON_20) as f:
    import_data_20 = json.load(f)

# Normalise all keys to str
import_data_20 = {
    str(k): {str(kk): v for kk, v in vv.items()}
    for k, vv in import_data_20.items()
}

print(f"Loaded import data for {len(import_data_20)} regions.")
print("Sample entry (region 3 -> others):")
for k, v in sorted(import_data_20.get("3", {}).items(), key=lambda x: -x[1])[:5]:
    print(f"  RIG {k}: {v:.4f}")

Loaded import data for 20 regions.
Sample entry (region 3 -> others):
  RIG 4: 0.5868
  RIG 14: 0.4681
  RIG 20: 0.3676
  RIG 19: 0.3396
  RIG 11: 0.2621


## 6. Generate yamls for all setups

Loads the `_DICE_CONSTANT` block from `region_yamls/default.yml` so every
generated file is a complete, loadable RICE yaml including the shared climate
model parameters.

In [6]:
DEFAULT_YAML_PATH = os.path.join(REPO_ROOT, "region_yamls", "default.yml")
with open(DEFAULT_YAML_PATH) as f:
    default_doc = yaml.safe_load(f)
DICE_BLOCK = default_doc.get("_DICE_CONSTANT", {})

def write_yaml(path, rice_params, dice_block):
    doc = {"_DICE_CONSTANT": dice_block, "_RICE_CONSTANT": rice_params}
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        yaml.dump(doc, f, default_flow_style=False, sort_keys=False)

generated = {}

for setup_name, setup_def in SETUPS.items():
    setup_dir = os.path.join(CBAM_DIR, setup_name)
    os.makedirs(setup_dir, exist_ok=True)
    cbam_map = build_cbam_index_map(setup_def["regions"])
    generated[setup_name] = {}

    print(f"\n{setup_name}: {setup_def['description']}")
    for cbam_idx, label, rigs in setup_def["regions"]:
        rice_params = merge_regions(rigs, params20, gdp20, import_data_20, cbam_map)
        out_path = os.path.join(setup_dir, f"{cbam_idx}.yml")
        write_yaml(out_path, rice_params, DICE_BLOCK)
        generated[setup_name][cbam_idx] = rice_params
        print(f"  {cbam_idx}. {label:<35}  "
              f"GDP={approx_gdp(rice_params):7.2f}T  "
              f"L0={rice_params['xL_0']:7.0f}M  "
              f"sigma={rice_params['xsigma_0']:.3f}")

print("\nAll yamls written to cbam_yamls/.")


setup_3: Minimal: EU | Merged CBAM Exporters | Rest of World
  1. EU & Western Europe                  GDP=  18.88T  L0=    502M  sigma=0.153
  2. CBAM Exporters (merged)              GDP=  27.66T  L0=   3658M  sigma=0.811
  3. Rest of World                        GDP=  44.17T  L0=   3501M  sigma=0.283

setup_5: Sector-split: EU | Eurasia | China | MENA | RoW
  1. EU & Western Europe                  GDP=  18.88T  L0=    502M  sigma=0.153
  2. Russia + Turkey + Eurasia            GDP=   3.19T  L0=    411M  sigma=0.936
  3. China                                GDP=  17.82T  L0=   1412M  sigma=0.784
  4. MENA (Gulf + N.Africa)               GDP=   3.50T  L0=    427M  sigma=0.782
  5. Rest of World                        GDP=  47.32T  L0=   4909M  sigma=0.322

setup_7: Granular: EU | Eurasia | China | India | Gulf | N.Africa | RoW
  1. EU & Western Europe                  GDP=  18.88T  L0=    502M  sigma=0.153
  2. Russia + Turkey + Eurasia            GDP=   3.19T  L0=    411M  sigma=0.9

## 7. Generate CountryClass CSVs

Each CSV maps every country from `CountryClass_20.csv` to its CBAM region index.
Required by `csv_asset/aggregate_local_mrio.py` for sector-level EORA aggregation.

In [7]:
cc20 = pd.read_csv(CC20_PATH)

for setup_name, setup_def in SETUPS.items():
    cbam_map = build_cbam_index_map(setup_def["regions"])
    label_map = {cbam_idx: label for cbam_idx, label, _ in setup_def["regions"]}

    cc_cbam = cc20.copy()
    cc_cbam["RIG"] = cc_cbam["RIG"].apply(
        lambda x: cbam_map.get(int(x), np.nan) if pd.notna(x) else np.nan
    )
    cc_cbam["RI"] = cc_cbam["RIG"].apply(
        lambda x: label_map.get(int(x), "") if pd.notna(x) else ""
    )

    n_regions = len(setup_def["regions"])
    out_path = os.path.join(CBAM_DIR, f"CountryClass_cbam_{n_regions}.csv")
    cc_cbam.to_csv(out_path, index=False)
    mapped = cc_cbam["RIG"].notna().sum()
    print(f"  {os.path.basename(out_path)}: {mapped} countries mapped across {n_regions} CBAM regions")

print("\nCountryClass CSVs written to cbam_yamls/.")

  CountryClass_cbam_3.csv: 216 countries mapped across 3 CBAM regions
  CountryClass_cbam_5.csv: 216 countries mapped across 5 CBAM regions
  CountryClass_cbam_7.csv: 216 countries mapped across 7 CBAM regions

CountryClass CSVs written to cbam_yamls/.


## 8. Summary — CBAM vulnerability indicators

For each setup: GDP, population, CO₂ emission intensity, and EU import weight
(how trade-connected each region is to the EU — higher = more CBAM exposure).

In [8]:
print("=" * 92)
for setup_name, setup_def in SETUPS.items():
    print(f"\n{'─'*92}")
    print(f"  {setup_name}: {setup_def['description']}")
    print(f"{'─'*92}")
    print(f"  {'#':>3}  {'Region':<35}  {'GDP (T$)':>9}  {'Pop (M)':>8}  "
          f"{'CO2 intensity':>13}  {'EU import wt':>12}")
    print(f"  {'─'*3}  {'─'*35}  {'─'*9}  {'─'*8}  {'─'*13}  {'─'*12}")

    eu_ximport = generated[setup_name][1]["ximport"]   # EU's import weights

    for cbam_idx, label, _ in setup_def["regions"]:
        p  = generated[setup_name][cbam_idx]
        eu_wt = eu_ximport.get(str(cbam_idx), 0.0)
        print(f"  {cbam_idx:>3}  {label:<35}  "
              f"{approx_gdp(p):>9.2f}  "
              f"{p['xL_0']:>8.0f}  "
              f"{p['xsigma_0']:>13.4f}  "
              f"{eu_wt:>12.4f}")

print(f"\n{'='*92}")
print()
print("Notes:")
print("  CO2 intensity (sigma): > 1.0 = high-carbon production relative to output")
print("  EU import weight: EU's ximport bid from that region (higher = stronger trade tie)")


────────────────────────────────────────────────────────────────────────────────────────────
  setup_3: Minimal: EU | Merged CBAM Exporters | Rest of World
────────────────────────────────────────────────────────────────────────────────────────────
    #  Region                                GDP (T$)   Pop (M)  CO2 intensity  EU import wt
  ───  ───────────────────────────────────  ─────────  ────────  ─────────────  ────────────
    1  EU & Western Europe                      18.88       502         0.1529        0.0000
    2  CBAM Exporters (merged)                  27.66      3658         0.8110        1.8792
    3  Rest of World                            44.17      3501         0.2827        2.2696

────────────────────────────────────────────────────────────────────────────────────────────
  setup_5: Sector-split: EU | Eurasia | China | MENA | RoW
────────────────────────────────────────────────────────────────────────────────────────────
    #  Region                          

## 9. Sanity check — Cobb-Douglas identity

For each generated yaml, verify that the back-calculated `xA_0` reproduces
the expected merged GDP exactly: `A * K^γ * (L/1000)^(1-γ) == sum(sub-region GDPs)`.

In [9]:
all_ok = True
for setup_name, setup_def in SETUPS.items():
    for cbam_idx, label, rigs in setup_def["regions"]:
        path = os.path.join(CBAM_DIR, setup_name, f"{cbam_idx}.yml")
        with open(path) as f:
            doc = yaml.safe_load(f)
        p = doc["_RICE_CONSTANT"]
        gdp_check = approx_gdp(p)
        expected  = sum(gdp20[r] for r in rigs if r in gdp20)
        rel_err   = abs(gdp_check - expected) / (expected + 1e-9)
        status    = "OK" if rel_err < 1e-6 else f"MISMATCH rel_err={rel_err:.2e}"
        if rel_err >= 1e-6:
            all_ok = False
        print(f"  {setup_name}/{cbam_idx}  [{label:<35}]  "
              f"check={gdp_check:.4f}  expected={expected:.4f}  {status}")

print()
print("All checks passed." if all_ok else "SOME CHECKS FAILED — see above.")

  setup_3/1  [EU & Western Europe                ]  check=18.8799  expected=18.8799  OK
  setup_3/2  [CBAM Exporters (merged)            ]  check=27.6558  expected=27.6558  OK
  setup_3/3  [Rest of World                      ]  check=44.1685  expected=44.1685  OK
  setup_5/1  [EU & Western Europe                ]  check=18.8799  expected=18.8799  OK
  setup_5/2  [Russia + Turkey + Eurasia          ]  check=3.1894  expected=3.1894  OK
  setup_5/3  [China                              ]  check=17.8205  expected=17.8205  OK
  setup_5/4  [MENA (Gulf + N.Africa)             ]  check=3.4956  expected=3.4956  OK
  setup_5/5  [Rest of World                      ]  check=47.3188  expected=47.3188  OK
  setup_7/1  [EU & Western Europe                ]  check=18.8799  expected=18.8799  OK
  setup_7/2  [Russia + Turkey + Eurasia          ]  check=3.1894  expected=3.1894  OK
  setup_7/3  [China                              ]  check=17.8205  expected=17.8205  OK
  setup_7/4  [India                   

## 10. Next steps

The yamls and CountryClass CSVs are now ready. Remaining steps for Phase 2:

1. **MRIO re-aggregation** — run `csv_asset/aggregate_local_mrio.py` with
   `CountryClass_cbam_N.csv` to produce aggregated EORA tables per CBAM setup.
   These provide sector CO₂ intensities and bilateral trade flow initialisation.

2. **Action space extension** (`rice_jax/_rice_mrio.py`) — add
   `export_reallocation` and `revenue_transfer` actions following the Phase 2A
   design documented in `rice_jax/rice_jax/MRIO_RICE_DESIGN.md`.

3. **Leakage elasticity calibration** — fit `ε_{r,s}` from MRIO cross-section
   data using JAX gradient-based optimisation.

4. **Validation** — run a dummy rollout per setup to confirm each environment
   initialises cleanly with the generated yamls.